# RandLANet Classification Debug with Pytorch Geometric

In [2]:
%load_ext autoreload
%autoreload 2

## Imports

In [3]:
import time
import os

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

## Open3D Blocks

In [4]:
class SharedMLP(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=1,
        stride=1,
        transpose=False,
        bn=True,
        activation_fn=None,
    ):
        super(SharedMLP, self).__init__()

        if transpose:
            self.conv = nn.ConvTranspose2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=(kernel_size - 1) // 2,
            )
        else:
            self.conv = nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=(kernel_size - 1) // 2,
            )

        self.batch_norm = (
            nn.BatchNorm2d(out_channels, eps=1e-6, momentum=0.01) if bn else None
        )
        self.activation_fn = activation_fn

    def forward(self, input):
        """Forward pass of the Module.

        Args:
            input: torch.Tensor of shape (B, dim_in, N, K)

        Returns:
            torch.Tensor, shape (B, dim_out, N, K)

        """
        x = self.conv(input)
        if self.batch_norm:
            x = self.batch_norm(x)
        if self.activation_fn:
            x = self.activation_fn(x)
        return x


class LocalSpatialEncoding(nn.Module):
    def __init__(self, dim_in, dim_out, num_neighbors, encode_pos=False):
        super(LocalSpatialEncoding, self).__init__()

        self.num_neighbors = num_neighbors
        self.mlp = SharedMLP(dim_in, dim_out, activation_fn=nn.LeakyReLU(0.2))
        self.encode_pos = encode_pos

    def gather_neighbor(self, coords, neighbor_indices):
        """Gather features based on neighbor indices.

        Args:
            coords: torch.Tensor of shape (B, N, d)
            neighbor_indices: torch.Tensor of shape (B, N, K)

        Returns:
            gathered neighbors of shape (B, dim, N, K)

        """
        B, N, K = neighbor_indices.size()
        dim = coords.shape[2]

        extended_indices = neighbor_indices.unsqueeze(1).expand(B, dim, N, K)
        extended_coords = coords.transpose(-2, -1).unsqueeze(-1).expand(B, dim, N, K)
        neighbor_coords = torch.gather(
            extended_coords, 2, extended_indices
        )  # (B, dim, N, K)

        return neighbor_coords

    def forward(self, coords, features, neighbor_indices, relative_features=None):
        """Forward pass of the Module.

        Args:
            coords: coordinates of the pointcloud
                torch.Tensor of shape (B, N, 3)
            features: features of the pointcloud.
                torch.Tensor of shape (B, d, N, 1)
            neighbor_indices: indices of k neighbours.
                torch.Tensor of shape (B, N, K)
            relative_features: relative neighbor features calculated
              on first pass. Required for second pass.

        Returns:
            torch.Tensor of shape (B, 2*d, N, K)

        """
        # finding neighboring points
        B, N, K = neighbor_indices.size()

        if self.encode_pos:
            neighbor_coords = self.gather_neighbor(coords, neighbor_indices)

            extended_coords = coords.transpose(-2, -1).unsqueeze(-1).expand(B, 3, N, K)

            relative_pos = extended_coords - neighbor_coords
            relative_dist = torch.sqrt(
                torch.sum(torch.square(relative_pos), dim=1, keepdim=True)
            )

            relative_features = torch.cat(
                [relative_dist, relative_pos, extended_coords, neighbor_coords], dim=1
            )

        else:
            if relative_features is None:
                raise ValueError(
                    "LocalSpatialEncoding: Require relative_features for second pass."
                )

        relative_features = self.mlp(relative_features)

        neighbor_features = self.gather_neighbor(
            features.transpose(1, 2).squeeze(3), neighbor_indices
        )

        return (
            torch.cat([neighbor_features, relative_features], dim=1),
            relative_features,
        )


class AttentivePooling(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(AttentivePooling, self).__init__()

        self.score_fn = nn.Sequential(
            nn.Linear(in_channels, in_channels), nn.Softmax(dim=-2)
        )
        self.mlp = SharedMLP(in_channels, out_channels, activation_fn=nn.LeakyReLU(0.2))

    def forward(self, x):
        """Forward pass of the Module.

        Args:
            x: torch.Tensor of shape (B, dim_in, N, K).

        Returns:
            torch.Tensor of shape (B, d_out, N, 1).

        """
        # computing attention scores
        scores = self.score_fn(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)

        # sum over the neighbors
        features = torch.sum(scores * x, dim=-1, keepdim=True)  # shape (B, d_in, N, 1)

        return self.mlp(features)


class LocalFeatureAggregation(nn.Module):
    def __init__(self, d_in, d_out, num_neighbors):
        super(LocalFeatureAggregation, self).__init__()

        self.num_neighbors = num_neighbors

        self.mlp1 = SharedMLP(d_in, d_out // 2, activation_fn=nn.LeakyReLU(0.2))
        self.lse1 = LocalSpatialEncoding(10, d_out // 2, num_neighbors, encode_pos=True)
        self.pool1 = AttentivePooling(d_out, d_out // 2)

        self.lse2 = LocalSpatialEncoding(d_out // 2, d_out // 2, num_neighbors)
        self.pool2 = AttentivePooling(d_out, d_out)
        self.mlp2 = SharedMLP(d_out, 2 * d_out)

        self.shortcut = SharedMLP(d_in, 2 * d_out)
        self.lrelu = nn.LeakyReLU()

    def forward(self, coords, feat, neighbor_indices):
        """Forward pass of the Module.

        Args:
            coords: coordinates of the pointcloud
                torch.Tensor of shape (B, N, 3).
            feat: features of the pointcloud.
                torch.Tensor of shape (B, d, N, 1)
            neighbor_indices: Indices of neighbors.

        Returns:
            torch.Tensor of shape (B, 2*d_out, N, 1).

        """
        x = self.mlp1(feat)

        x, neighbor_features = self.lse1(coords, x, neighbor_indices)
        x = self.pool1(x)

        x, _ = self.lse2(
            coords, x, neighbor_indices, relative_features=neighbor_features
        )
        x = self.pool2(x)

        return self.lrelu(self.mlp2(x) + self.shortcut(feat))

## Torch PointCloud Implementation

### Test LocalFeatureAggregation

In [108]:
from torch_pointcloud.models.randlanet import LocalFeatureAggregation as LocalFeatureAggregation_TP
from torch_pointcloud.ops import knn, knn_interpolate

In [109]:
xyz = torch.rand(32, 1024, 3)
features = torch.rand(32, 3, 1024).unsqueeze(-1)
dists, idxs = knn(xyz, xyz, k=16)

In [110]:
lfa = LocalFeatureAggregation(3, 16, num_neighbors=16)

out_original = lfa(xyz, features, idxs)
print(f"{out_original.shape = }")

out_original.shape = torch.Size([32, 32, 1024, 1])


In [111]:
lfa_tp = LocalFeatureAggregation_TP(3, 16, num_neighbors=16)

out_tp = lfa_tp(xyz, features)
print(f"{out_tp.shape = }")

RuntimeError: Given groups=1, weight of size [8, 8, 1, 1], expected input[32, 10, 1024, 16] to have 8 channels, but got 10 channels instead

In [94]:
state_dict_mapping = {
    'mlp1.conv.weight': 'mlp1.convs.0.weight',
    'mlp1.conv.bias': 'mlp1.convs.0.bias',
    'mlp1.batch_norm.weight': 'mlp1.norms.0.weight',
    'mlp1.batch_norm.bias': 'mlp1.norms.0.bias',
    'mlp1.batch_norm.running_mean': 'mlp1.norms.0.running_mean',
    'mlp1.batch_norm.running_var': 'mlp1.norms.0.running_var',
    'mlp1.batch_norm.num_batches_tracked': 'mlp1.norms.0.num_batches_tracked',
    'mlp2.conv.weight': 'mlp2.convs.0.weight',
    'mlp2.conv.bias': 'mlp2.convs.0.bias',
    'mlp2.batch_norm.weight': 'mlp2.norms.0.weight',
    'mlp2.batch_norm.bias': 'mlp2.norms.0.bias',
    'mlp2.batch_norm.running_mean': 'mlp2.norms.0.running_mean',
    'mlp2.batch_norm.running_var': 'mlp2.norms.0.running_var',
    'mlp2.batch_norm.num_batches_tracked': 'mlp2.norms.0.num_batches_tracked',
    'shortcut.conv.weight': 'mlp_skip.convs.0.weight',
    'shortcut.conv.bias': 'mlp_skip.convs.0.bias',
    'shortcut.batch_norm.weight': 'mlp_skip.norms.0.weight',
    'shortcut.batch_norm.bias': 'mlp_skip.norms.0.bias',
    'shortcut.batch_norm.running_mean': 'mlp_skip.norms.0.running_mean',
    'shortcut.batch_norm.running_var': 'mlp_skip.norms.0.running_var',
    'shortcut.batch_norm.num_batches_tracked': 'mlp_skip.norms.0.num_batches_tracked',
    'lse1.mlp.conv.weight': 'lse1.mlp.convs.0.weight',
    'lse1.mlp.conv.bias': 'lse1.mlp.convs.0.bias',
    'lse1.mlp.batch_norm.weight': 'lse1.mlp.norms.0.weight',
    'lse1.mlp.batch_norm.bias': 'lse1.mlp.norms.0.bias',
    'lse1.mlp.batch_norm.running_mean': 'lse1.mlp.norms.0.running_mean',
    'lse1.mlp.batch_norm.running_var': 'lse1.mlp.norms.0.running_var',
    'lse1.mlp.batch_norm.num_batches_tracked': 'lse1.mlp.norms.0.num_batches_tracked',
    'lse2.mlp.conv.weight': 'lse2.mlp.convs.0.weight',
    'lse2.mlp.conv.bias': 'lse2.mlp.convs.0.bias',
    'lse2.mlp.batch_norm.weight': 'lse2.mlp.norms.0.weight',
    'lse2.mlp.batch_norm.bias': 'lse2.mlp.norms.0.bias',
    'lse2.mlp.batch_norm.running_mean': 'lse2.mlp.norms.0.running_mean',
    'lse2.mlp.batch_norm.running_var': 'lse2.mlp.norms.0.running_var',
    'lse2.mlp.batch_norm.num_batches_tracked': 'lse2.mlp.norms.0.num_batches_tracked',
    'pool1.score_fn.0.weight': 'pool1.attn.0.weight',
    'pool1.score_fn.0.bias': 'pool1.attn.0.bias',
    'pool1.mlp.conv.weight': 'pool1.mlp.convs.0.weight',
    'pool1.mlp.conv.bias': 'pool1.mlp.convs.0.bias',
    'pool1.mlp.batch_norm.weight': 'pool1.mlp.norms.0.weight',
    'pool1.mlp.batch_norm.bias': 'pool1.mlp.norms.0.bias',
    'pool1.mlp.batch_norm.running_mean': 'pool1.mlp.norms.0.running_mean',
    'pool1.mlp.batch_norm.running_var': 'pool1.mlp.norms.0.running_var',
    'pool1.mlp.batch_norm.num_batches_tracked': 'pool1.mlp.norms.0.num_batches_tracked',
    'pool2.score_fn.0.weight': 'pool2.attn.0.weight',
    'pool2.score_fn.0.bias': 'pool2.attn.0.bias',
    'pool2.mlp.conv.weight': 'pool2.mlp.convs.0.weight',
    'pool2.mlp.conv.bias': 'pool2.mlp.convs.0.bias',
    'pool2.mlp.batch_norm.weight': 'pool2.mlp.norms.0.weight',
    'pool2.mlp.batch_norm.bias': 'pool2.mlp.norms.0.bias',
    'pool2.mlp.batch_norm.running_mean': 'pool2.mlp.norms.0.running_mean',
    'pool2.mlp.batch_norm.running_var': 'pool2.mlp.norms.0.running_var',
    'pool2.mlp.batch_norm.num_batches_tracked': 'pool2.mlp.norms.0.num_batches_tracked',
}


lfa_state_dict = lfa.state_dict()
lfa_tp_state_dict = lfa_tp.state_dict()

for key, value in lfa_state_dict.items():
    tp_key = state_dict_mapping.get(key)
    assert lfa_tp_state_dict[tp_key].shape == lfa_state_dict[key].shape, f"{tp_key = }, {key = }"
    lfa_tp_state_dict[tp_key] = lfa_state_dict[key]

AssertionError: tp_key = 'lse2.mlp.convs.0.weight', key = 'lse2.mlp.conv.weight'

In [103]:
lfa.lse1

LocalSpatialEncoding(
  (mlp): SharedMLP(
    (conv): Conv2d(10, 8, kernel_size=(1, 1), stride=(1, 1))
    (batch_norm): BatchNorm2d(8, eps=1e-06, momentum=0.01, affine=True, track_running_stats=True)
    (activation_fn): LeakyReLU(negative_slope=0.2)
  )
)

In [102]:
lfa_tp.lse1

LocalSpatialEncoding(
  (mlp): SharedMLP(
    (convs): ModuleList(
      (0): Conv2d(10, 8, kernel_size=(1, 1), stride=(1, 1))
    )
    (norms): ModuleList(
      (0): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (act): LeakyReLU(negative_slope=0.2)
  )
)

In [93]:
lfa_tp_state_dict["lse2.mlp.convs.0.weight"].shape

torch.Size([8, 10, 1, 1])

In [95]:
lfa_state_dict["lse2.mlp.conv.weight"].shape

torch.Size([8, 8, 1, 1])